# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Colab T4: lấy mã nguồn và cài thư viện (không dùng Kaggle Notebook/API).
!nvidia-smi -L
%cd /content
!if [ ! -d road-damage-lab/.git ]; then git clone --depth 1 https://github.com/VinUni-AI20k/K4-Track-4-Day10-VisionLanguageModel.git road-damage-lab; fi
%cd /content/road-damage-lab
!grep -vi '^kaggle$' requirements-colab.txt > /content/requirements-colab-no-kaggle.txt
!pip -q install -r /content/requirements-colab-no-kaggle.txt
from google.colab import drive
from pathlib import Path
import shutil
try:
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/day10_qwen_e234')
    DRIVE_WORK = DRIVE_ROOT / 'work'
    DRIVE_EVIDENCE = DRIVE_ROOT / 'evidence'
    DRIVE_WORK.mkdir(parents=True, exist_ok=True)
    DRIVE_EVIDENCE.mkdir(parents=True, exist_ok=True)
    print('Checkpoint/outputs Drive:', DRIVE_ROOT)
except Exception as exc:
    DRIVE_ROOT = DRIVE_WORK = DRIVE_EVIDENCE = None
    print('Drive mount failed; run stays in this Colab session:', repr(exc))

### Dữ liệu (không cần Kaggle)
Cell tiếp theo dùng dữ liệu đã có, tìm trong Drive/thư mục runtime, hoặc hiện nút tải ZIP lên Colab nếu chưa thấy dataset. ZIP cần chứa `images/` và `labels-YOLO/`.


## 1. Dữ liệu: tải và chia split

In [ ]:
%cd /content/road-damage-lab
from pathlib import Path
import json, shutil, subprocess, sys, urllib.request
if all(Path(f'data/splits/{s}.jsonl').exists() for s in ('train','val','test')):
    print('Split đã tồn tại; giữ nguyên để resume.')
else:
    api = 'https://zenodo.org/api/records/17834373'
    with urllib.request.urlopen(api, timeout=60) as response:
        record = json.load(response)
    assert record['title'] == 'Road Damage Dataset: Potholes, Cracks and Manholes', record.get('title')
    archive = next(item for item in record['files'] if item['key'] == 'archive.zip')
    zip_path = Path('/content/archive.zip')
    if not zip_path.exists() or zip_path.stat().st_size != archive['size']:
        print(f"Tải {archive['key']} ({archive['size']/1e6:.1f} MB) từ Zenodo...")
        urllib.request.urlretrieve(archive['links']['self'], zip_path)
    assert zip_path.stat().st_size == archive['size'], 'ZIP tải về chưa đủ dung lượng'
    subprocess.run([sys.executable, 'scripts/prepare_data.py', '--zip', str(zip_path)], check=True)
if DRIVE_ROOT is not None:
    shutil.copytree('data', DRIVE_ROOT / 'data', dirs_exist_ok=True)
    print('Đã lưu data/splits lên Drive:', DRIVE_ROOT / 'data')


In [ ]:
# Kiểm tra split và trực quan hoá 6 ảnh (hộp màu theo lớp).
import json, random
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

splits = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}
meta = json.loads(Path("data/splits/meta.json").read_text(encoding="utf-8"))
for name, rows in splits.items():
    print(name, "images=", len(rows), "boxes=", sum(len(r["objects"]) for r in rows))
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not (sessions["train"] & sessions["val"] or sessions["train"] & sessions["test"] or sessions["val"] & sessions["test"])
print("dataset_version:", meta["dataset_version"], "; session leakage: none")

COLORS = {"pothole": "red", "crack": "yellow", "manhole": "lime"}
for row in random.Random(42).sample(splits["train"], min(6, len(splits["train"]))):
    image_path = Path(row["image_path"])
    if not image_path.exists(): image_path = Path("data") / image_path
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    for obj in row["objects"]:
        xc, yc, w, h = obj["yolo"]
        box = ((xc-w/2)*row["width"], (yc-h/2)*row["height"], (xc+w/2)*row["width"], (yc+h/2)*row["height"])
        draw.rectangle(box, outline=COLORS[obj["label"]], width=2)
        draw.text((box[0], max(0, box[1]-12)), obj["label"], fill=COLORS[obj["label"]])
    print(row["image_id"]); display(image)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json, math, os, re, time, random, contextlib
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import LoraConfig, PeftModel, get_peft_model

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
CLASSES = ("pothole", "crack", "manhole")
PROMPT = "Detect all road damages in this image. Classes: pothole, crack, manhole. Return only a JSON list, each item with label and bbox_2d [x1,y1,x2,y2] as integers on the 0-1000 coordinate scale relative to image width and height. Return [] if none."
PIXEL_PROMPT = "Detect all road damages in this image. Classes: pothole, crack, manhole. Return only a JSON list, each item with label and bbox_2d [x1,y1,x2,y2] as pixel integers relative to the 640x360 image. Return [] if none."

def yolo_to_box(label, yolo, width=640, height=360, coord="norm1000"):
    xc, yc, w, h = map(float, yolo)
    scale_x, scale_y = ((1000, 1000) if coord == "norm1000" else (width, height))
    vals = ((xc-w/2)*scale_x, (yc-h/2)*scale_y, (xc+w/2)*scale_x, (yc+h/2)*scale_y)
    x1, y1, x2, y2 = [int(max(0, min(limit, round(v)))) for v, limit in zip(vals, (scale_x, scale_y, scale_x, scale_y))]
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects, coord="norm1000"):
    boxes = [yolo_to_box(o["label"], o["yolo"], coord=coord) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"), ensure_ascii=False)

print(objects_to_answer(splits["train"][0]["objects"]))


## 3. E0: zero-shot

In [ ]:
# Nạp model QLoRA NF4. Giữ toàn bộ visual và lm_head ở dense precision.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, llm_int8_skip_modules=['model.visual', 'visual', 'lm_head'])
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'left'
# Dùng nhất quán ảnh 320x180 cho E1-E4 để vừa T4 và so sánh công bằng.
processor.image_processor.min_pixels = 320 * 180
processor.image_processor.max_pixels = 320 * 180
device = torch.device('cuda:0')

def load_base():
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0})
    import bitsandbytes as bnb
    assert not any(isinstance(x, bnb.nn.Linear4bit) for x in model.model.visual.modules()), 'visual must remain dense'
    model.config.use_cache = False
    return model

def image_path(row):
    path = Path(row['image_path'])
    return path if path.exists() else Path('data') / path

@torch.inference_mode()
def predict_rows(model, rows, prompt=PROMPT, batch_size=1, max_new_tokens=256):
    model.eval(); output = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start+batch_size]
        messages = [[{'role':'user', 'content':[{'type':'image', 'image':Image.open(image_path(r)).convert('RGB')}, {'type':'text', 'text':prompt}]}] for r in batch]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                    return_dict=True, return_tensors='pt', padding=True).to(device)
        with torch.autocast('cuda', dtype=torch.float16):
            generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        prompt_len = inputs['input_ids'].shape[1]
        decoded = processor.batch_decode(generated[:, prompt_len:], skip_special_tokens=True)
        output.extend({'image_id': r['image_id'], 'raw': raw.strip()} for r, raw in zip(batch, decoded))
    return output

def write_jsonl(path, rows):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row, ensure_ascii=False) + '\n')
print('Model/inference helpers ready. E0 is skipped; the pipeline below starts at E1.')

In [ ]:
# E0 đã có kết quả trong REPORT.md. Không chạy lại để bắt đầu trực tiếp từ E1.
RUN_E0 = False
if RUN_E0:
    base = load_base()
    e0_predictions = predict_rows(base, splits['test'])
    write_jsonl('work/e0.jsonl', e0_predictions)
    del base, e0_predictions
    torch.cuda.empty_cache()
    !python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
def make_messages(row, coord="norm1000"):
    prompt = PROMPT if coord == "norm1000" else PIXEL_PROMPT
    image = Image.open(image_path(row)).convert("RGB")
    user = {"role":"user", "content":[{"type":"image", "image":image}, {"type":"text", "text":prompt}]}
    return user, {"role":"assistant", "content":objects_to_answer(row["objects"], coord=coord)}

def encode_example(row, coord="norm1000"):
    user, answer = make_messages(row, coord)
    prompt_enc = processor.apply_chat_template([user], tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    full_enc = processor.apply_chat_template([user, answer], tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt")
    n = prompt_enc["input_ids"].shape[1]
    assert torch.equal(full_enc["input_ids"][0, :n], prompt_enc["input_ids"][0]), "prompt tokens are not a prefix; inspect chat template"
    labels = full_enc["input_ids"].clone(); labels[:, :n] = -100
    full_enc["labels"] = labels
    return full_enc

def collate(examples):
    pad_id = processor.tokenizer.pad_token_id
    max_len = max(x["input_ids"].shape[1] for x in examples)
    keys = ("input_ids", "attention_mask", "mm_token_type_ids", "labels")
    out = {}
    for key in keys:
        vals = [x[key] for x in examples if key in x]
        if not vals: continue
        fill = -100 if key == "labels" else (pad_id if key == "input_ids" else 0)
        padded = []
        for value in vals:
            amount = max_len - value.shape[1]
            padded.append(torch.nn.functional.pad(value, (0, amount), value=fill))
        out[key] = torch.cat(padded, dim=0)
    for key in ("pixel_values", "image_grid_thw"):
        vals = [x[key] for x in examples if key in x]
        if vals: out[key] = torch.cat(vals, dim=0)
    return out

sample = encode_example(splits["train"][0])
print("image patches/grid:", sample.get("image_grid_thw"), "supervised tokens:", int((sample["labels"] != -100).sum()))
assert "pixel_values" in sample and "image_grid_thw" in sample


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
def apply_setup(model, setup, rank=16):
    for p in model.parameters(): p.requires_grad_(False)
    model.enable_input_require_grads()
    llm = r'(?:model\.)?language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))'
    vit = r'(?:model\.)?visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))'
    target = llm if setup in ('E1', 'E2') else f'({llm})|({vit})'
    names = [n for n, _ in model.named_modules() if re.fullmatch(target, n)]
    expected = 252 if setup in ('E1', 'E2') else 348
    assert len(names) == expected, f'LoRA target count {len(names)} != {expected}; inspect model.named_modules()'
    model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=2*rank, lora_dropout=0.05,
                          target_modules=target, task_type='CAUSAL_LM'))
    if setup in ('E2', 'E3'):
        found = 0
        for name, param in model.named_parameters():
            if 'visual.merger' in name or 'visual.deepstack_merger_list' in name:
                param.data = param.data.float(); param.requires_grad_(True); found += param.numel()
        assert found >= 100_000_000, f'projector trainable size unexpectedly small: {found}'
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(setup, 'matched modules=', len(names), 'trainable params=', f'{trainable:,}')
    return model, trainable

## 6. Vòng huấn luyện

In [ ]:
import copy, gc, time, random, numpy as np, torch

GRAD_ACCUM = 8
MICRO_BATCH = 1
SAVE_EVERY = 100

def _sync_run(run_dir):
    if 'DRIVE_WORK' not in globals() or DRIVE_WORK is None: return
    dest = DRIVE_WORK / Path(run_dir).name
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)

def _restore_run_dir(run_dir):
    if 'DRIVE_WORK' not in globals() or DRIVE_WORK is None: return
    src = DRIVE_WORK / Path(run_dir).name
    if not (Path(run_dir) / 'trainer.pt').exists() and src.exists() and any(src.iterdir()):
        shutil.copytree(src, run_dir, dirs_exist_ok=True)
        print('Restored saved run/checkpoint from Drive:', src)

def _save_model_parts(model, run_dir):
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(run_dir / 'adapter')
    extra = {n:p.detach().float().cpu() for n,p in model.named_parameters()
             if p.requires_grad and ('visual.merger' in n or 'visual.deepstack_merger_list' in n)}
    if extra: torch.save(extra, run_dir / 'extra_trainable.pt')

def train_experiment(setup, rank=16, train_fraction=1.0, max_steps=300, seed=42):
    run_dir = Path(f'work/{setup.lower()}_r{rank}'); run_dir.mkdir(parents=True, exist_ok=True)
    _restore_run_dir(run_dir)
    rows = list(splits['train']); random.Random(seed).shuffle(rows)
    rows = rows[:max(1, int(len(rows)*train_fraction))]

    state_path = run_dir / 'trainer.pt'
    completed_path = run_dir / 'run.json'
    if completed_path.exists() and (run_dir / 'adapter').exists():
        prior = json.loads(completed_path.read_text(encoding='utf-8'))
        if prior.get('pipeline_version') == 2 and prior.get('max_steps') == max_steps and prior.get('rank') == rank:
            print('Using completed run:', run_dir)
            return run_dir, prior
    if state_path.exists() and (run_dir / 'adapter').exists():
        model = load_base()
        model = PeftModel.from_pretrained(model, run_dir / 'adapter', is_trainable=True)
        extra_path = run_dir / 'extra_trainable.pt'
        if extra_path.exists():
            extra = torch.load(extra_path, map_location='cpu', weights_only=True)
            missing, unexpected = model.load_state_dict(extra, strict=False)
            assert not unexpected, unexpected
            for name, param in model.named_parameters():
                if 'visual.merger' in name or 'visual.deepstack_merger_list' in name:
                    param.data = param.data.float(); param.requires_grad_(True)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print('Resuming', setup, 'from optimizer step', json.loads((run_dir/'progress.json').read_text())['step'])
    else:
        model, trainable = apply_setup(load_base(), setup, rank=rank)

    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    model.config.use_cache = False
    train_params = [p for p in model.parameters() if p.requires_grad]
    lr = 2e-4 if setup == 'E1' else 2e-5
    optimizer = torch.optim.AdamW(train_params, lr=lr, weight_decay=0, foreach=False)
    warmup = max(1, int(max_steps * 0.05))
    def lr_lambda(step):
        if step < warmup: return (step + 1) / warmup
        progress = min(1.0, (step - warmup) / max(1, max_steps - warmup))
        return 0.5 * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler('cuda')

    step = epoch = cursor = 0; losses = []; elapsed_before = 0.0; peak_before = 0.0
    if state_path.exists():
        state = torch.load(state_path, map_location='cpu', weights_only=False)
        if state.get('version') != 2: raise RuntimeError('Checkpoint version mismatch; remove the incomplete run folder and rerun this setup.')
        optimizer.load_state_dict(state['optimizer']); scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler'])
        step, epoch, cursor = state['step'], state['epoch'], state['cursor']
        losses, elapsed_before, peak_before = state['losses'], state['elapsed_seconds'], state['peak_vram_gb']
        random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng'].cpu()); torch.cuda.set_rng_state_all([x.cpu() for x in state['cuda_rng']])
    else:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    optimizer.zero_grad(set_to_none=True); micro = 0; started = time.time()
    torch.cuda.reset_peak_memory_stats()
    while step < max_steps:
        order = torch.randperm(len(rows), generator=torch.Generator().manual_seed(seed + epoch)).tolist()
        if cursor >= len(order): epoch += 1; cursor = 0; continue
        idx = order[cursor:cursor+MICRO_BATCH]; cursor += len(idx)
        batch = collate([encode_example(rows[i]) for i in idx])
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.autocast('cuda', dtype=torch.float16): loss = model(**batch).loss / GRAD_ACCUM
        if not torch.isfinite(loss): raise FloatingPointError(f'non-finite loss at optimizer step {step}')
        scaler.scale(loss).backward(); losses.append(float(loss.detach()) * GRAD_ACCUM); micro += 1
        if micro == GRAD_ACCUM:
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(train_params, 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
            step += 1; micro = 0
            if step % 25 == 0: print(setup, 'rank', rank, 'step', step, 'loss=', round(sum(losses[-100:])/len(losses[-100:]), 4))
            if step % SAVE_EVERY == 0 or step == max_steps:
                _save_model_parts(model, run_dir)
                progress = {'setup':setup,'rank':rank,'step':step,'max_steps':max_steps,'mean_last_loss':sum(losses[-100:])/len(losses[-100:])}
                (run_dir/'progress.json').write_text(json.dumps(progress, indent=2), encoding='utf-8')
                state = {'version':2,'step':step,'epoch':epoch,'cursor':cursor,'losses':losses,
                         'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
                         'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
                         'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
                         'elapsed_seconds':elapsed_before + time.time()-started,
                         'peak_vram_gb':max(peak_before, torch.cuda.max_memory_allocated()/1024**3)}
                torch.save(state, state_path); _sync_run(run_dir)

    minutes = (elapsed_before + time.time()-started) / 60
    peak = max(peak_before, torch.cuda.max_memory_allocated()/1024**3)
    run = {'pipeline_version':2,'setup':setup,'rank':rank,'trainable_params':trainable,'peak_vram_gb':peak,
           'train_minutes':minutes,'max_steps':step,'mean_last_loss':sum(losses[-100:])/len(losses[-100:]),
           'resolution_pixels':320*180,'dataset_version':json.loads(Path('data/splits/meta.json').read_text())['dataset_version']}
    _save_model_parts(model, run_dir)
    (run_dir/'run.json').write_text(json.dumps(run, indent=2), encoding='utf-8')
    if state_path.exists(): state_path.unlink()
    if 'DRIVE_WORK' in globals() and DRIVE_WORK is not None:
        saved_state = DRIVE_WORK / run_dir.name / 'trainer.pt'
        if saved_state.exists(): saved_state.unlink()
    _sync_run(run_dir)
    print('Completed:', run)
    del model, optimizer, scheduler, scaler; gc.collect(); torch.cuda.empty_cache()
    return run_dir, run

def train_all():
    return {setup: train_experiment(setup, rank=16, max_steps=300) for setup in ('E1','E2','E3')}


## 7. Suy luận và chấm E1–E3

In [ ]:
def load_trained(run_dir, setup):
    model = load_base()
    model = PeftModel.from_pretrained(model, Path(run_dir)/'adapter', is_trainable=False)
    extra_path = Path(run_dir)/'extra_trainable.pt'
    if extra_path.exists():
        state = torch.load(extra_path, map_location='cpu', weights_only=True)
        missing, unexpected = model.load_state_dict(state, strict=False)
        assert not unexpected, unexpected
        assert not [n for n in missing if 'visual.merger' in n or 'visual.deepstack_merger_list' in n]
    return model

def evaluate_run(run_dir, setup, split, evidence_name):
    run_dir = Path(run_dir); model = load_trained(run_dir, setup)
    preds = predict_rows(model, splits[split], batch_size=1)
    pred_path = Path('work') / f'{evidence_name}.jsonl'; write_jsonl(pred_path, preds)
    del model, preds; gc.collect(); torch.cuda.empty_cache()
    run = json.loads((run_dir/'run.json').read_text(encoding='utf-8'))
    out = Path('report/evidence') / f'{evidence_name}.json'; out.parent.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, 'scripts/score.py', '--pred', str(pred_path), '--split', split, '--name', evidence_name, '--out', str(out),
           '--meta', f"trainable_params={run['trainable_params']}", '--meta', f"peak_vram_gb={run['peak_vram_gb']:.5f}",
           '--meta', f"train_minutes={run['train_minutes']:.5f}"]
    subprocess.run(cmd, check=True)
    if 'DRIVE_EVIDENCE' in globals() and DRIVE_EVIDENCE is not None: shutil.copy2(out, DRIVE_EVIDENCE/out.name)
    return json.loads(out.read_text(encoding='utf-8'))


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 chọn đúng một biến: rank LoRA r=4 và r=64, alpha=2r.
def run_e4_rank(setup_best):
    results = {}
    for rank in (4, 64):
        run_dir, run = train_experiment(setup_best, rank=rank, max_steps=300)
        val = evaluate_run(run_dir, setup_best, 'val', f'e4_rank_{rank}_val')
        results[rank] = {'run_dir':run_dir, 'run':run, 'val':val}
    return results


## 9. Chạy liền mạch E1 → E4
Chạy cell cuối sau các cell setup và dữ liệu. Cell này tiếp tục từ checkpoint Drive nếu runtime Colab bị ngắt. E0 được giữ nguyên kết quả cũ.


In [ ]:
import subprocess, sys, json, shutil, gc, torch
from pathlib import Path

assert torch.cuda.is_available(), 'Hãy bật GPU T4 trong Runtime → Change runtime type.'
assert all(Path(f'data/splits/{s}.jsonl').exists() for s in ('train','val','test')), 'Chưa có dữ liệu; chạy cell tải dataset Zenodo/split trước.'
Path('report/evidence').mkdir(parents=True, exist_ok=True)

# Train/resume E1, E2, E3. Checkpoint và optimizer state được lưu lên Drive mỗi 100 bước.
runs = train_all()

# Chọn setup tốt nhất chỉ bằng validation. Chưa đụng đến test ở bước chọn mô hình.
val_scores = {}
for setup, (run_dir, run) in runs.items():
    result = evaluate_run(run_dir, setup, 'val', f'{setup.lower()}_val')
    val_scores[setup] = result['micro']['f1']
print('Validation F1:', val_scores)
setup_best = max(val_scores, key=val_scores.get)
print('E4 base selected from validation:', setup_best)

# Huấn luyện hai rank của E4 và so sánh validation.
e4_runs = run_e4_rank(setup_best)
e4_val = {rank: item['val']['micro']['f1'] for rank, item in e4_runs.items()}
print('E4 validation F1:', e4_val)

# Sau khi đã chốt toàn bộ lựa chọn bằng validation, chấm test cho E1-E3 và hai mức rank E4.
for setup, (run_dir, run) in runs.items():
    evaluate_run(run_dir, setup, 'test', setup.lower())
for rank, item in e4_runs.items():
    evaluate_run(item['run_dir'], setup_best, 'test', f'e4_rank_{rank}')

summary = {'validation_f1':val_scores, 'e4_validation_f1':e4_val, 'e4_base_setup':setup_best,
           'test_evidence':[f'report/evidence/{s.lower()}.json' for s in runs] +
                           [f'report/evidence/e4_rank_{r}.json' for r in (4,64)]}
Path('work/e1234_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
if 'DRIVE_EVIDENCE' in globals(): shutil.copy2('work/e1234_summary.json', DRIVE_EVIDENCE/'e1234_summary.json')
print('E1-E4 completed:', json.dumps(summary, ensure_ascii=False, indent=2))
